In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk(r'C:\Users\shrey\OneDrive\Documents\python\it_project\Cattle Breeds'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.image import imread
import cv2
import random
import os
from os import listdir
from PIL import Image
import tensorflow as tf
from keras.preprocessing import image
from tensorflow.keras.utils import img_to_array, array_to_img
from keras.optimizers import Adam
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D
from keras.layers import Activation, Flatten, Dropout, Dense
from sklearn.model_selection import train_test_split
from keras.models import model_from_json
from keras.utils import to_categorical
from tensorflow.keras.callbacks import ReduceLROnPlateau
from keras.callbacks import EarlyStopping

In [ ]:
plt.figure(figsize=(12,12))
path=r"C:\Users\shrey\OneDrive\Documents\python\it_project\Cattle Breeds"
breeds=os.listdir(path)
for i in range(1,17):
    plt.subplot(4,4,i)
    plt.tight_layout()
    random_breed = random.choice(breeds)

    # Pick a random image from that breed folder
    random_image = random.choice(os.listdir(path + "/" + random_breed))

    # Full path
    img_path = path + "/" + random_breed + "/" + random_image

    # Read and show
    rand_img = imread(img_path)
    plt.xlabel(rand_img.shape[1],fontsize=10)
    plt.ylabel(rand_img.shape[0],fontsize=10)
    plt.imshow(rand_img)


In [ ]:
def convert_img_to_array(image_dir):
    try:
        image=cv2.imread(image_dir)
        if image is not None:
            image=cv2.resize(image,(128,128))
            return img_to_array(image)
        else:
            np.array([])
    except Exception as e:
        print(f"Error,{e}")
        return None

In [ ]:
dir=r"C:\Users\shrey\OneDrive\Documents\python\it_project\Cattle Breeds"
image_list,label_list=[],[]
all_labels=['Ayrshire cattle','Brown Swiss cattle','Holstein Friesian cattle', 'Jersey cattle', 'Red Dane cattle']
binary_labels=[0,1,2,3,4]
temp=-1
for directory in all_labels:
    cattle_image_list=listdir(f"{dir}/{directory}")
    temp+=1
    count=0
    for files in cattle_image_list:
        image_path=f"{dir}/{directory}/{files}"
        image_list.append(convert_img_to_array(image_path))
        label_list.append(binary_labels[temp])
        count=count+1
        if(count==204):
            break


In [ ]:
label_counts=pd.DataFrame(label_list).value_counts()
label_counts.head()

In [ ]:
image_list[0].shape

In [ ]:

x_train,x_test,y_train,y_test=train_test_split(image_list,label_list,test_size=0.2,random_state=10)


In [ ]:
x_train=np.array(x_train,dtype=np.float16)/255
x_test=np.array(x_test,dtype=np.float16)/255
x_train=x_train.reshape(-1,128,128,3)
x_test=x_test.reshape(-1,128,128,3)

In [ ]:
y_train=to_categorical(y_train)
y_test=to_categorical(y_test)

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest',
    rescale=1./255,

)

datagen.fit(x_train)


In [ ]:
x_train,x_val,y_train,y_val=train_test_split(x_train,y_train,test_size=0.2,random_state=10)
print(x_train.shape, y_train.shape)
print(x_val.shape, y_val.shape)


In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

# Load base model (or start fresh)
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(128,128,3))
base_model.trainable = False
for layer in base_model.layers[-20:]:
    layer.trainable = True

# Build the full model on top
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu', kernel_regularizer=l2(0.001))(x)
x = Dropout(0.5)(x)
output = Dense(5, activation='softmax')(x)

model2 = Model(inputs=base_model.input, outputs=output)
model2.compile(optimizer=Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])

# Save the full model safely
#model1.save('trained_model.keras')


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True
)

history=model2.fit(x_train,y_train,epochs=100,validation_data=[x_val,y_val])

In [ ]:
loss, accuracy = model2.evaluate(x_test, y_test)
print(f'Test Loss: {loss:.4f}')
print(f'Test Accuracy: {accuracy:.4f}')

In [ ]:
hist=model2.history

In [ ]:
import matplotlib.pyplot as plt
epochs=[i for i in range(1,101)]
plt.plot(epochs,hist.history['accuracy'],label='Training Accuracy')
plt.plot(epochs,hist.history['val_accuracy'],label='Validation Accuracy')
plt.legend()
plt.show()

In [ ]:
test_Set=x_val
y_pred=model2.predict(test_Set)
y_pred,y_pred.shape

In [ ]:
predicted_categories=tf.argmax(y_pred,axis=1)
predicted_categories

In [ ]:
if len(y_val.shape) > 1 and y_val.shape[1] > 1:  # one-hot encoded
    true_categories = np.argmax(y_val, axis=1)
else:
    true_categories = y_val  # already in label form

print("True categories:", true_categories)

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(true_categories,predicted_categories))

In [ ]:
#saving the model
model2.save('trained_model.keras')

In [ ]:
#Recording history
import json
with open('history.json', 'w') as f:
    json.dump(hist.history, f)

In [ ]:
model = tf.keras.models.load_model('c:\Users\shrey\Downloads\trained_model_2.h5', safe_mode=False)# to check layers
